# Week4_ex3 - Lorentz Force Between Two Parallel Wires

Two long parallel current-carrying wires push or pull on each other through the Lorentz force -- this notebook builds the geometry, solves it as Magnetostatic, and cross-checks the field results against the textbook infinite-wire B-field formula.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import matplotlib.pyplot as plt


In [ ]:
# 1. AEDT desktop interface
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. turn off autosave
DT.disable_autosave()

# 3. create the project and set the solution type
sol_type = "Magnetostatic"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object, needed to reuse script-recording snippets
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Week4_ex3"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week4_ex3"


In [ ]:
## Save project and apply design name ##

proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()


In [ ]:
# Helper for finding the two end faces of a coil object -- picks the faces with the
# largest |x| coordinate and returns them sorted (outer face first, inner face second)
def find_terminal_face(winding_obj):
    terminal_face = []

    # find the face with the maximum x position on this object
    max_x_pos = max(abs(face.center[0]) for face in winding_obj.faces)

    # collect every face sitting at that x position
    for face in winding_obj.faces:
        if abs(abs(face.center[0]) - max_x_pos) <= 0.0001:
            terminal_face.append(face)

    # sort so the outer face comes first
    ter_out, ter_in = sorted(terminal_face, key=lambda x: x.center[0], reverse=True)

    return ter_out, ter_in


In [ ]:
line_length = 500
M3D["line_length"] = f"{line_length}mm"

line_diameter = 10
M3D["line_diameter"] = f"{line_diameter}mm"

line_distance = 50
M3D["line_distance"] = f"{line_distance}mm"

line_num_seg = 12
M3D["line_num_seg"] = f"{line_num_seg}"

Current = 100
M3D["Current"] = f"{Current}A"


In [ ]:
## Create and configure analysis setup ##
# (Magnetostatic solve, so the plain create_setup() call works fine here)

my_setup = M3D.create_setup(name="Setup1")

my_setup.props["MaximumPasses"] = 20
my_setup.props["PercentRefinement"] = 15
my_setup.props["PercentError"] = 1
my_setup.props["RelativeResidual"] = 1e-06
my_setup.props["NonLinearResidual"] = 0.001
my_setup.props["SmoothBHCurve"] = False
my_setup.props["MuOption"] = {"MuNonLinearBH": True}
my_setup.update()


In [ ]:
# Draw the two straight wires as polylines with a circular cross-section
point_tmp = []
point_tmp.append(["-line_length/2", "-line_distance/2", 0])
point_tmp.append(["line_length/2", "-line_distance/2", 0])
line_1 = M3D.modeler.create_polyline(points=point_tmp, name="line_1", material="copper", xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)

# a zero-thickness line running right along wire 1, just for sampling field data later
field_line_1 = M3D.modeler.create_polyline(points=point_tmp, name="field_line_1")

# repeat for the second wire, offset by line_distance
point_tmp = []
point_tmp.append(["-line_length/2", "line_distance/2", 0])
point_tmp.append(["line_length/2", "line_distance/2", 0])
line_2 = M3D.modeler.create_polyline(points=point_tmp, name="line_2", material="copper", xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)

field_line_2 = M3D.modeler.create_polyline(points=point_tmp, name="field_line_2")

# background region for the solve
region = M3D.modeler.create_region(pad_value=[0, 0, 800, 800, 6000, 6000], pad_type='Percentage Offset', name='Region')


In [ ]:
# NOTE: MaxLength coarsened from 5mm to 15mm, and setup's PercentRefinement
# dropped from 30 to 15 -- at 5mm/30% this pushed the region's adaptive mesh to
# ~68,870 elements, over the Student license's cap. Coarser wire mesh + slower
# per-pass refinement keeps it under the cap while still smoothing the field plot.
oModule = oDesign.GetModule("MeshSetup")

oModule.AssignLengthOp(
	[
		"NAME:Length1",
		"RefineInside:=", True,
		"Enabled:=", True,
		"Objects:=", ["line_1","line_2"],
		"RestrictElem:=", False,
		"NumMaxElem:=", "1000",
		"RestrictLength:=", True,
		"MaxLength:=", "15mm"
	])


In [ ]:
# Grab the two end faces of each wire to use as coil terminals
ter_line1_in = line_1.bottom_face_x
ter_line1_out = line_1.top_face_x

ter_line2_in = line_2.bottom_face_x
ter_line2_out = line_2.top_face_x

M3D.assign_coil(assignment=ter_line1_in, conductors_number=1, polarity="Positive", name="in_1")
M3D.assign_coil(assignment=ter_line1_out, conductors_number=1, polarity="Negative", name="out_1")

M3D.assign_coil(assignment=ter_line2_in, conductors_number=1, polarity="Positive", name="in_2")
M3D.assign_coil(assignment=ter_line2_out, conductors_number=1, polarity="Negative", name="out_2")

# Bundle each pair of coil terminals into its own current winding
winding_1 = M3D.assign_winding(assignment=None, winding_type="Current", is_solid=True, current="Current", 
                               resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_1")

winding_2 = M3D.assign_winding(assignment=None, winding_type="Current", is_solid=True, current="Current", 
                               resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_2")

M3D.add_winding_coils(assignment=winding_1.name, coils=["in_1", "out_1"])
M3D.add_winding_coils(assignment=winding_2.name, coils=["in_2", "out_2"])


In [ ]:
# Need to register a force parameter before AEDT will report force on an object.
# Lorentz force (not virtual) works fine here since both wires are just copper, mu_r ~ 1.
M3D.assign_force(assignment=line_1, coordinate_system='Global', is_virtual=False, force_name="force_line1")

M3D.assign_force(assignment=line_2, coordinate_system='Global', is_virtual=False, force_name="force_line2")


In [ ]:
## analyze ##

my_setup.analyze()

In [ ]:
## Report -- force on each wire vs. wire length ##

oModule = oDesign.GetModule("ReportSetup")
oModule.CreateReport("Force Table 1", "Magnetostatic", "Data Table", "Setup1 : LastAdaptive", [], 
	[
		"line_length:=", ["All"],
		"line_diameter:=", ["Nominal"],
		"line_distance:=", ["Nominal"],
		"line_num_seg:=", ["Nominal"],
		"Current:=", ["Nominal"]
	], 
	[
		"Y Component:=", ["force_line1.Force_x","force_line1.Force_y","force_line1.Force_z","force_line2.Force_x","force_line2.Force_y","force_line2.Force_z"]
	])
oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Force Table 1:force_line1.Force_x:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:=", "newton"
				]
			]
		]
	])
oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Force Table 1:force_line1.Force_y:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:=", "newton"
				]
			]
		]
	])
oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Force Table 1:force_line1.Force_z:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:=", "newton"
				]
			]
		]
	])
oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Force Table 1:force_line2.Force_x:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:=", "newton"
				]
			]
		]
	])
oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Force Table 1:force_line2.Force_y:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:=", "newton"
				]
			]
		]
	])
oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Force Table 1:force_line2.Force_z:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:=", "newton"
				]
			]
		]
	])

csv_name = f"{proj_name}_force_table.csv"
dir_csv = csv_name  # saved flat, next to the notebook -- same convention as Images/
oModule.ExportToFile("Force Table 1", dir_csv, False)


In [ ]:
## Load the force data back in ##

# show 3 decimal places instead of pandas' default float precision
pd.options.display.float_format = '{:.3f}'.format

df = pd.read_csv(dir_csv)

df_filtered_line1 = df[['force_line1.Force_x [newton]', 'force_line1.Force_y [newton]', 'force_line1.Force_z [newton]']]
df_filtered_line2 = df[['force_line2.Force_x [newton]', 'force_line2.Force_y [newton]', 'force_line2.Force_z [newton]']]

display(df_filtered_line1)
display(df_filtered_line2)


In [ ]:
## Check the field produced by wire 2 alone (B_2) ##

M3D.cleanup_solution(variations='All', entire_solution=True, field=True, mesh=True, linked_data=True)

# turn off the current in wire 1, keep wire 2 driven
winding_1["Current"] = "0"
display(winding_1.props)
winding_2["Current"] = "Current"
display(winding_2.props)

# re-solve with this configuration
my_setup.analyze()

# sample B along field_line_1, which sits right where wire 1 used to be driven
oModule = oDesign.GetModule("ReportSetup")
oModule.CreateReport("B_2", "Fields", "Rectangular Plot", "Setup1 : LastAdaptive", 
	[
		"Context:=", "field_line_1",
		"PointCount:=", 501
	], 
	[
		"Distance:=", ["All"],
		"line_length:=", ["Nominal"],
		"line_diameter:=", ["Nominal"],
		"line_distance:=", ["Nominal"],
		"line_num_seg:=", ["Nominal"],
		"Current:=", ["Nominal"]
	], 
	[
		"X Component:=", "Distance",
		"Y Component:=", ["Mag_B"]
	])

csv_name = "Week4_ex3_B2.csv"
dir_csv_b2 = csv_name  # saved flat, next to the notebook -- same convention as Images/
oModule.ExportToFile("B_2", dir_csv_b2, False)


In [ ]:
## Check the field produced by wire 1 alone (B_1) ##

M3D.cleanup_solution(variations='All', entire_solution=True, field=True, mesh=True, linked_data=True)

# now flip it: wire 1 carries current, wire 2 is off
winding_1["Current"] = "Current"
display(winding_1.props)
winding_2["Current"] = "0"
display(winding_2.props)

my_setup.analyze()

# sample B along field_line_2, which sits where wire 2 used to be driven
oModule = oDesign.GetModule("ReportSetup")
oModule.CreateReport("B_1", "Fields", "Rectangular Plot", "Setup1 : LastAdaptive", 
	[
		"Context:=", "field_line_2",
		"PointCount:=", 501
	], 
	[
		"Distance:=", ["All"],
		"line_length:=", ["Nominal"],
		"line_diameter:=", ["Nominal"],
		"line_distance:=", ["Nominal"],
		"line_num_seg:=", ["Nominal"],
		"Current:=", ["Nominal"]
	], 
	[
		"X Component:=", "Distance",
		"Y Component:=", ["Mag_B"]
	])

csv_name = "Week4_ex3_B1.csv"
dir_csv_b1 = csv_name  # saved flat, next to the notebook -- same convention as Images/
oModule.ExportToFile("B_1", dir_csv_b1, False)


In [ ]:
## Plot B_1 and B_2 alongside the analytic infinite-wire formula ##

b1_df = pd.read_csv(dir_csv_b1)
b2_df = pd.read_csv(dir_csv_b2)

# grab column headers so the units can be pulled out below
head_b1 = list(b1_df.columns)
head_b2 = list(b2_df.columns)

unit_len = head_b1[0].replace("Distance [", "").replace("]", "")
unit_T = head_b1[1].replace("Mag_B [", "").replace("]", "")

length_data = b1_df[head_b1[0]].values
b1_data = b1_df[head_b1[1]].values
b2_data = b2_df[head_b2[1]].values

avg_b1 = np.mean(b1_data)
avg_b2 = np.mean(b2_data)

mu0 = 4.0 * np.pi * 1e-7  # vacuum permeability [H/m]

# analytic B field of an infinite wire, evaluated at the wire spacing, for comparison
b1_formula = mu0 * Current / (2 * math.pi * (line_distance * 1e-3)) * 1e+6  # uTesla
b2_formula = mu0 * Current / (2 * math.pi * (line_distance * 1e-3)) * 1e+6  # uTesla

plt.figure(figsize=(8,6))
plt.plot(length_data, b1_data, 'r-', linewidth=5, label='B_1')
plt.plot(length_data, b2_data, 'b-', linewidth=5, label='B_2')
plt.xlabel(f'line position ({unit_len})')
plt.ylabel(f'B ({unit_T})')
plt.title(' ')
plt.legend()
plt.grid(True)
plt.ylim(avg_b1/5*4, avg_b1/5*6)

textstr = f"B_1 average: {avg_b1:.2f} uT\nB_1 formula: {b1_formula:.2f} uT\n\nB_2 average: {avg_b2:.2f} uT\nB_2 formula: {b2_formula:.2f} uT\n"
plt.text(0.05, 0.95, textstr, transform=plt.gca().transAxes, 
         verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))

os.makedirs("Images", exist_ok=True)
plt.savefig("Images/Week4_ex3_B_field_vs_position.png")
plt.show()


In [ ]:
## save project ##

M3D.save_project()